# Many recordings, one dataset

This notebook prepares several recordings with one configuration, combines
them into a single dataset with an identifier per epoch, and exports it.
It shows what the consistency check catches, how to keep going past a
failing recording, and how the identifiers support leakage-free splits
downstream. No model is trained: that is the user's code, not this
library's.

The recordings are synthetic (3 subjects x 2 sessions) so the notebook runs
anywhere.

In [ ]:
import tempfile
from pathlib import Path

import mne
import numpy as np

from eeg_decoder.io import to_raw

mne.set_log_level("warning")
CHANNELS = ["F3", "Fz", "F4", "C3", "Cz", "C4", "P3", "Pz", "P4", "O1", "Oz", "O2"]


def synthetic_recording(seed, sfreq=250.0, seconds=60.0):
    """12 EEG channels (microvolts) and a stim channel: 1 = standard, 2 = target."""
    rng = np.random.default_rng(seed)
    n = int(seconds * sfreq)
    spectrum = np.fft.rfft(rng.normal(size=(3, n)), axis=1)
    spectrum[:, 1:] /= np.sqrt(np.fft.rfftfreq(n, 1 / sfreq)[1:])  # 1/f background
    spectrum[:, 0] = 0
    latent = np.fft.irfft(spectrum, n, axis=1)
    mixing = rng.normal(1.0, 0.3, (len(CHANNELS), 3))  # shared sources: correlated channels
    latent = latent / latent.std(axis=1, keepdims=True) * 6
    eeg = mixing @ latent + rng.normal(0, 2, (len(CHANNELS), n))
    onsets = np.arange(int(2 * sfreq), n - int(2 * sfreq), int(1.5 * sfreq))
    codes = np.where(rng.random(onsets.size) < 0.25, 2, 1)
    p300 = 8 * np.exp(-((np.arange(int(0.8 * sfreq)) / sfreq - 0.35) / 0.08) ** 2)
    for onset in onsets[codes == 2]:
        eeg[CHANNELS.index("Pz"), onset:onset + p300.size] += p300
    stim = np.zeros(n)
    stim[onsets] = codes
    return to_raw(np.vstack([eeg, stim]), sfreq, CHANNELS + ["STI"],
                  ["eeg"] * len(CHANNELS) + ["stim"], units="uV")


recordings = [
    (synthetic_recording(seed=10 * s + k), {"subject": f"sub-0{s}", "session": session})
    for s in (1, 2, 3) for k, session in enumerate(("A", "B"))
]
[meta for _, meta in recordings]

## 1. One configuration for every recording

`prepare_recordings` runs the same `PreprocessingConfig` on each
`(raw, metadata)` pair, checks that the results are compatible, and
combines them. `ch_types=["eeg"]` keeps only EEG channels in the arrays.

In [ ]:
from eeg_decoder.dataset import prepare_recordings
from eeg_decoder.preprocessing import PreprocessingConfig

config = PreprocessingConfig([
    {"op": "bandpass", "l_freq": 0.5, "h_freq": 30.0},
    {"op": "rereference", "ref": "average"},
    {"op": "make_epochs", "event_source": "stim", "stim_channel": "STI",
     "event_id": {"standard": 1, "target": 2}, "tmin": -0.2, "tmax": 0.8},
    {"op": "baseline_correct", "baseline": [None, 0.0]},
    {"op": "reject_epochs", "reject": {"eeg": 150e-6}},
])
ds = prepare_recordings(recordings, config, ch_types=["eeg"])
print(ds.summary())

Everything downstream code needs is in one object. `X` is in volts and
read-only; every per-epoch array stays aligned with `X` after rejection.

In [ ]:
print("X", ds.X.shape, ds.X.dtype, "| sfreq", ds.sfreq, "| times", ds.times[[0, -1]])
print("y", ds.y[:8], ds.event_id)
print("subject", ds.groups["subject"][:3], "... session", ds.groups["session"][:3], "...")
print(len(ds.records), "pipeline records, one per recording")

## 2. What the consistency check catches

A recording made at 500 Hz yields epochs with twice as many samples.
Instead of resampling it silently, combining stops and lists **every**
difference at once.

In [ ]:
odd = (synthetic_recording(seed=99, sfreq=500.0), {"subject": "sub-04", "session": "A"})
try:
    prepare_recordings(recordings[:1] + [odd], config, ch_types=["eeg"])
except ValueError as err:
    print(err)

Resample it explicitly (add `{"op": "resample", "sfreq": 250.0}` for that
recording) or leave it out. To keep going past recordings that fail,
collect the failures instead of stopping at the first:

In [ ]:
# A recording saved without its stim channel: no events to epoch.
no_stim = (recordings[0][0].copy().drop_channels(["STI"]), {"subject": "sub-05", "session": "A"})
ds_collected = prepare_recordings(recordings + [no_stim], config, ch_types=["eeg"],
                                  on_error="collect")
kept = np.unique(ds_collected.recording)
print(ds_collected.n_epochs, "epochs from recordings", kept.tolist())
# Indices keep referring to your input list: records has one entry per input,
# None where the recording failed.
print([r is None for r in ds_collected.records])
for failure in ds_collected.failures:
    print(f"recording {failure['index']} {dict(failure['metadata'])}:")
    print(f"  {failure['error']}: {failure['message']}")

## 3. Export

`.npz` stores plain arrays and JSON only (it loads with
`allow_pickle=False`) and round-trips exactly. Per-recording MNE `.fif`
files keep the full `Epochs` objects, in double precision and anonymized,
with a JSON sidecar holding the metadata and the processing record.

In [ ]:
from eeg_decoder.dataset import load_epochs_fif, load_npz, save_npz

with tempfile.TemporaryDirectory() as tmp:
    path = save_npz(ds, Path(tmp) / "dataset.npz")
    back = load_npz(path)
    print("exact round trip:", np.array_equal(back.X, ds.X) and back.records == ds.records)

    prepare_recordings(recordings, config, ch_types=["eeg"], fif_dir=tmp)
    print(sorted(p.name for p in Path(tmp).glob("recording-00[01]*")))
    epochs, sidecar = load_epochs_fif(Path(tmp) / "recording-000-epo.fif")
    print(sidecar["metadata"], "| anonymized:", sidecar["anonymized"], "|", epochs)

## 4. Leakage-free splits downstream

Epochs from one subject are not independent, so a model must be tested on
subjects it never saw. The per-epoch identifiers make that a one-liner,
with NumPy or with scikit-learn's `GroupKFold` / `LeaveOneGroupOut`
(`groups=ds.groups["subject"]`). Anything *fitted* to the data (scaling,
CSP, the model) belongs inside that loop, on the training part only.

In [ ]:
subjects = ds.groups["subject"]
for held_out in np.unique(subjects):
    test = subjects == held_out
    print(f"test on {held_out}: train {np.count_nonzero(~test)} epochs, "
          f"test {np.count_nonzero(test)} epochs")